# TradeSense · Component 03 · Data audit of all layers (read only)

Checks, with numbers, whether the three data layers and the order-book data are fit for the system: Layer 1 (1-minute bars: continuity, duplicates, sanity of prices and volumes, hash of every daily file), Layer 2 (Sapienza pump-and-dump and Mt.Gox wash trading: label rates, split order, overlap between splits), Layer 3 (synthetic events: counts, split placement), the Ardia and Bluteau pump list, and the time coverage of the public 2026 order-book data. It also prints the exact feature definitions of Layer 1 so that Layer 1 can be rebuilt from the Tardis trades for the August to October days.

**Read only.** Nothing is written except `data_audit_v1/` (a report and a share block). CPU runtime, about 5 to 15 minutes. Every section is protected: if one fails, the others still run and the failure is listed.

In [ ]:
import os, re, json, time, glob, hashlib, warnings, traceback
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir()
OUT = BASE / 'data_audit_v1'; OUT.mkdir(exist_ok=True)
RES = {}; FAILS = []; LINES = []
def say(s=''): print(s); LINES.append(str(s))
def section(name):
    def deco(fn):
        try: fn()
        except Exception as e: FAILS.append(name); say(f'!! section {name} failed: {type(e).__name__}: {str(e)[:200]}'); traceback.print_exc(limit=2)
        return fn
    return deco
def sha(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

Mounted at /content/drive


## Layer 1 · 1-minute bars

In [ ]:
@section('layer1')
def _():
    d = BASE / 'processed_layer1_final_v1' / 'daily_cache'; files = sorted(d.glob('*_l1_*.parquet')); say(f'[Layer 1] daily files: {len(files)}'); rows = []; hashbad = 0; metabad = 0; frames = {}
    for p in files:
        m = re.match(r'(\w+)_l1_(\d{4}-\d{2}-\d{2})\.parquet', p.name); sym, day = m.group(1), m.group(2); df = pd.read_parquet(p); t = pd.to_datetime(df['datetime'], utc=True); dt = t.diff().dropna().dt.total_seconds()
        meta_p = p.with_name(p.stem + '_meta.json'); ok_hash = None; ok_val = None
        if meta_p.exists():
            mj = json.loads(meta_p.read_text()); ok_val = bool(mj.get('validation_passed')); ok_hash = (mj.get('parquet_sha256') == sha(p)) if mj.get('parquet_sha256') else None; hashbad += int(ok_hash is False); metabad += int(ok_val is False)
        o, h_, l_, c = (df[k].astype(float) for k in ('open', 'high', 'low', 'close')); feat = [c_ for c_ in df.columns if c_ not in ('datetime',)]
        rows.append({'sym': sym, 'day': day, 'rows': len(df), 'gaps>60s': int((dt > 60).sum()), 'dup_ts': int(t.duplicated().sum()), 'nan': int(df[feat].isna().sum().sum()), 'inf': int(np.isinf(df[feat].select_dtypes('number')).sum().sum()), 'ohlc_bad': int(((h_ < np.maximum(o, c) - 1e-9) | (l_ > np.minimum(o, c) + 1e-9)).sum()), 'neg_vol': int((df['volume'] < 0).sum()), 'zero_vol_min': int((df['volume'] == 0).sum()), 'hash_ok': ok_hash, 'validation_passed': ok_val})
        if day[-2:] == '01' or True: frames.setdefault(sym, []).append(df[['datetime', 'log_returns', 'volume', 'trades_count', 'trade_flow_imbalance', 'hl_spread_ratio', 'large_trade_count']])
    T = pd.DataFrame(rows); RES['layer1_days'] = int(len(T)); say(f'  symbols {sorted(T.sym.unique())}; days per symbol {T.groupby("sym").size().to_dict()}; range {T.day.min()} -> {T.day.max()}')
    say(f'  days with rows != 1440: {int((T.rows != 1440).sum())} | minute gaps: {int(T["gaps>60s"].sum())} | duplicate timestamps: {int(T.dup_ts.sum())} | NaN cells: {int(T.nan.sum())} | inf: {int(T.inf.sum())} | OHLC violations: {int(T.ohlc_bad.sum())} | negative volume: {int(T.neg_vol.sum())} | zero-volume minutes: {int(T.zero_vol_min.sum())}')
    say(f'  hash mismatches: {hashbad} | validation_passed False: {metabad}')
    RES['layer1'] = {'days': len(T), 'range': [T.day.min(), T.day.max()], 'rows_bad_days': int((T.rows != 1440).sum()), 'gaps': int(T['gaps>60s'].sum()), 'dups': int(T.dup_ts.sum()), 'nan': int(T.nan.sum()), 'ohlc_bad': int(T.ohlc_bad.sum()), 'hash_mismatch': hashbad, 'validation_failed': metabad}
    for sym, fl in frames.items():
        A = pd.concat(fl); A['m'] = pd.to_datetime(A['datetime'], utc=True).dt.to_period('M'); g = A.groupby('m').agg(minutes=('volume', 'size'), vol_mean=('volume', 'mean'), ret_std=('log_returns', 'std'), trades_mean=('trades_count', 'mean'), tfi_mean=('trade_flow_imbalance', 'mean'), large_mean=('large_trade_count', 'mean')); say(f'  {sym}: monthly summary'); say((g.round(5)).to_string())
    for nm in ('pilot_feature_definitions.json', 'thresholds_calibrated.json'):
        q = BASE / 'processed_layer1_final_v1' / 'metadata' / nm
        if q.exists(): say(f'  --- {nm}'); say(json.dumps(json.loads(q.read_text()), indent=1)[:3500])
    RES['layer1_end_day'] = T.day.max()

[Layer 1] daily files: 180
  symbols ['BTCUSDT', 'ETHUSDT']; days per symbol {'BTCUSDT': 90, 'ETHUSDT': 90}; range 2024-05-01 -> 2024-07-29
  days with rows != 1440: 0 | minute gaps: 0 | duplicate timestamps: 0 | NaN cells: 0 | inf: 0 | OHLC violations: 0 | negative volume: 0 | zero-volume minutes: 0
  hash mismatches: 180 | validation_passed False: 0
  BTCUSDT: monthly summary
         minutes  vol_mean  ret_std  trades_mean  tfi_mean  large_mean
m                                                                     
2024-05    44640  21.17005  0.00064    906.53156  -0.03220    55.14014
2024-06    43200  16.13005  0.00051    818.09382  -0.02265    37.22347
2024-07    41760  20.64767  0.00072   1238.40802  -0.02999    46.98381
  ETHUSDT: monthly summary
         minutes   vol_mean  ret_std  trades_mean  tfi_mean  large_mean
m                                                                      
2024-05    44640  259.83519  0.00086    647.37092  -0.00200    33.46156
2024-06    43200  176

## Layer 2 · Sapienza and Mt.Gox

In [ ]:
@section('layer2')
def _():
    d = BASE / 'processed_layer2_final_v3'; say(f'[Layer 2] {d.name}: ' + ', '.join(sorted(p.name for p in d.iterdir())))
    for p in sorted(d.glob('*.parquet')):
        df = pd.read_parquet(p); lab = [c for c in df.columns if c in ('wash', 'target', 'gt', 'label', 'y')]; row = {'file': p.name, 'rows': len(df), 'cols': len(df.columns), 'nan_cells': int(df.isna().sum().sum())}
        if lab: row['label'] = lab[0]; row['positive_rate'] = float(df[lab[0]].mean()); row['positives'] = int(df[lab[0]].sum())
        say('  ' + json.dumps(row)); RES.setdefault('layer2_files', []).append(row)
    sd = BASE / 'processed_layer2_model_ready_v1' / 'sapienza' / 'metadata'; sp = {}
    for s in ('train', 'val', 'test'):
        q = sd / f'sapienza_{s}_sidecar.parquet'
        if q.exists(): sp[s] = pd.read_parquet(q)
    if sp:
        rows = []
        for s, df in sp.items(): rows.append({'split': s, 'rows': len(df), 'positives': int((df['gt'] == 1).sum()), 'positive_rate': float((df['gt'] == 1).mean()), 'events(pos groups)': int(df.loc[df['gt'] == 1, 'event_group_id'].nunique()), 'coins': int(df.coin.nunique()), 'first': str(df.date.min())[:10], 'last': str(df.date.max())[:10]})
        say('  Sapienza splits'); say((pd.DataFrame(rows)).to_string())
        ev = {s: set(df.loc[df['gt'] == 1, 'event_group_id']) for s, df in sp.items()}; vh = {s: set(df.vec_hash) for s, df in sp.items() if 'vec_hash' in df}
        say('  event groups shared between splits: ' + ', '.join(f'{a}&{b}={len(ev[a] & ev[b])}' for a in ev for b in ev if a < b));
        if vh: say('  identical feature vectors shared between splits: ' + ', '.join(f'{a}&{b}={len(vh[a] & vh[b])}' for a in vh for b in vh if a < b))
        say('  chronological order train < val < test: ' + str(bool(pd.to_datetime(sp["train"].date).max() <= pd.to_datetime(sp["val"].date).min() and pd.to_datetime(sp["val"].date).max() <= pd.to_datetime(sp["test"].date).min())) if len(sp) == 3 else '')
        RES['sapienza'] = rows
    for sub in ('mtgox', 'sapienza'):
        q = BASE / 'processed_layer2_model_ready_v1' / sub / 'reports'
        for j in sorted(q.glob('*.json'))[:2]:
            try: o = json.loads(j.read_text()); say(f'  --- {sub}/{j.name} keys: {list(o)[:12]}'); [say(f'      {k}: {json.dumps(o[k])[:260]}') for k in ('classification', 'verdict', 'reasons', 'chronology_supported') if k in o]
            except Exception as e: say(f'  could not read {j.name}')

[Layer 2] processed_layer2_final_v3: mtgox_metadata.joblib, mtgox_test.parquet, mtgox_train.parquet, mtgox_val.parquet, sapienza_metadata.joblib, sapienza_test.parquet, sapienza_train.parquet, sapienza_val.parquet
  {"file": "mtgox_test.parquet", "rows": 830588, "cols": 8, "nan_cells": 0, "label": "wash", "positive_rate": 0.2088436144032902, "positives": 173463}
  {"file": "mtgox_train.parquet", "rows": 3876076, "cols": 8, "nan_cells": 0, "label": "wash", "positive_rate": 0.39349254245788784, "positives": 1525207}
  {"file": "mtgox_val.parquet", "rows": 830588, "cols": 8, "nan_cells": 0, "label": "wash", "positive_rate": 0.24558023954114436, "positives": 203976}
  {"file": "sapienza_test.parquet", "rows": 89334, "cols": 16, "nan_cells": 0, "label": "gt", "positive_rate": 0.000537309423064007, "positives": 48}
  {"file": "sapienza_train.parquet", "rows": 600827, "cols": 16, "nan_cells": 0, "label": "gt", "positive_rate": 0.00036616197341331197, "positives": 220}
  {"file": "sapienza_val

## Layer 3 · synthetic events

In [ ]:
@section('layer3')
def _():
    d = BASE / 'processed_layer3_final_v2'; ev = pd.concat([pd.read_parquet(p) for p in sorted((d / 'events').glob('*_events.parquet'))]); say(f'[Layer 3 v2] events: {len(ev)}')
    t = ev.groupby(['symbol', 'split']).size().unstack(fill_value=0); say('  events per symbol and split'); say((t).to_string())
    say('  scenario families (all):'); say((ev.groupby('scenario_family').size().to_frame('events').T).to_string())
    say('  duration bars: ' + json.dumps(ev.duration_bars.describe().round(1).to_dict()) + ' | severity: ' + json.dumps(ev.severity.describe().round(2).to_dict()) + f' | OOD share {ev.is_ood.mean():.2f} | composite share {ev.composite.mean():.2f}')
    sm = json.loads((d / 'metadata' / 'split_manifest.json').read_text()); say('  split bounds: ' + json.dumps(sm.get('bounds_bar_index'))[:300]); say('  rows per symbol/split: ' + json.dumps(sm.get('rows_per_symbol_split'))[:300])
    bad = 0
    for sym in ('btc', 'eth'):
        e = ev[ev.symbol == sym.upper() + 'USDT'];
        for s, g in e.groupby('split'):
            f = d / sym / f'{s}_state.parquet'
            if f.exists():
                tt = pd.to_datetime(pd.read_parquet(f, columns=['datetime'])['datetime'], utc=True); lo, hi = tt.min(), tt.max(); st = pd.to_datetime(g.start_timestamp, utc=True); en = pd.to_datetime(g.end_timestamp, utc=True); bad += int(((st < lo) | (en > hi)).sum())
    say(f'  events lying outside the time range of their split file: {bad}'); RES['layer3'] = {'events': int(len(ev)), 'outside_split': bad, 'by_split': {str(k): v for k, v in ev.groupby('split').size().to_dict().items()}}
    vr = json.loads((d / 'validation' / 'validation_report.json').read_text()); say('  validation verdict: ' + str(vr.get('verdict')) + ' | scientific gates: ' + json.dumps(vr.get('scientific_gates'))[:400])
    lim = d / 'reports' / 'limitations.md'
    if lim.exists(): say('  --- limitations.md (first 1500 chars)'); say(lim.read_text()[:1500])

[Layer 3 v2] events: 292
  events per symbol and split
split    test  train  val
symbol                   
BTCUSDT    36     80   30
ETHUSDT    36     80   30
  scenario families (all):
scenario_family  abnormal_volume_burst  composite  fake_breakout_like  liquidity_stress_proxy  pump_and_dump_like  trade_flow_imbalance  wash_like_aggregate_proxy
events                              44         30                  46                      42                  46                    42                         42
  duration bars: {"count": 292.0, "mean": 32.7, "std": 24.1, "min": 5.0, "25%": 16.0, "50%": 25.0, "75%": 42.0, "max": 144.0} | severity: {"count": 292, "unique": 3, "top": "subtle", "freq": 114} | OOD share 0.06 | composite share 0.10
  split bounds: {"train": [0, 90720], "val": [90720, 109440], "test": [109440, 129600]}
  rows per symbol/split: {"train": 90720, "val": 18720, "test": 20160}
  events lying outside the time range of their split file: 0
  validation verdict: READY WITH

## Ardia and Bluteau pump list

In [ ]:
@section('ardia')
def _():
    p = BASE / 'Ardia_Bluteau_PumpDump_1160' / 'data' / 'list_pd_events.csv'; df = pd.read_csv(p); df['y'] = pd.to_datetime(df['pump_date'], utc=True, errors='coerce').dt.year; say(f'[Ardia & Bluteau list] rows {len(df)}, unique pump dates {df.pump_date.nunique()}, years {df.y.min()}-{df.y.max()}')
    say('  by exchange:'); say((df.groupby('Exchange').size().sort_values(ascending=False).head(8).to_frame('events').T).to_string()); say('  by year:'); say((df.groupby('y').size().to_frame('events').T).to_string()); say(f'  success rate {df.success.mean():.2f}; Binance rows: {int((df.Exchange.astype(str).str.lower() == "binance").sum())}')
    RES['ardia'] = {'rows': int(len(df)), 'binance': int((df.Exchange.astype(str).str.lower() == "binance").sum())}

[Ardia & Bluteau list] rows 1160, unique pump dates 306, years 2019.0-2022.0
  by exchange:
Exchange  Binance
events       1160
  by year:
y       2019.0  2020.0  2021.0  2022.0
events     130      92      97       3
  success rate 0.28; Binance rows: 1160


## Public 2026 order-book data

In [ ]:
@section('public2026')
def _():
    import pyarrow.parquet as pq
    root = BASE / 'TGNN_Public_OrderBook_Dataset'; rows = []
    for p in sorted(root.rglob('*.parquet')):
        f = pq.ParquetFile(p); names = f.schema_arrow.names; ci = names.index('timestamp_ms'); lo = None; hi = None
        for g in range(f.metadata.num_row_groups):
            st = f.metadata.row_group(g).column(ci).statistics
            if st is not None and st.has_min_max: lo = st.min if lo is None else min(lo, st.min); hi = st.max if hi is None else max(hi, st.max)
        rows.append({'file': str(p.relative_to(root)), 'rows': f.metadata.num_rows, 'from_utc': str(pd.to_datetime(lo, unit='ms', utc=True)) if lo else None, 'to_utc': str(pd.to_datetime(hi, unit='ms', utc=True)) if hi else None, 'MB': round(p.stat().st_size / 1e6)})
    T = pd.DataFrame(rows); say('[Public order-book dataset 2026]'); say((T).to_string()); RES['public2026'] = rows
    rd = root / 'README.md'
    if rd.exists(): say('  --- README (first 1800 chars)'); say(rd.read_text()[:1800])

[Public order-book dataset 2026]
                                         file      rows                          from_utc                            to_utc   MB
0       depth/binance/BTCUSDT/2026-06.parquet  68905191  2026-06-04 00:54:40.378000+00:00  2026-06-30 23:58:26.423000+00:00  386
1       depth/binance/BTCUSDT/2026-07.parquet  50583603  2026-07-01 00:54:17.803000+00:00  2026-07-31 23:56:46.326000+00:00  318
2       depth/binance/ETHUSDT/2026-06.parquet  45852893  2026-06-04 00:54:40.378000+00:00  2026-06-30 23:58:26.424000+00:00  312
3       depth/binance/ETHUSDT/2026-07.parquet  38398343  2026-07-01 00:54:17.803000+00:00  2026-07-31 23:56:46.327000+00:00  283
4       depth/binance/SOLUSDT/2026-06.parquet   9948461  2026-06-04 00:54:40.378000+00:00  2026-06-30 23:58:26.404000+00:00  110
5       depth/binance/SOLUSDT/2026-07.parquet   6893242  2026-07-01 00:54:17.803000+00:00  2026-07-31 23:56:46.207000+00:00   83
6   snapshots/binance/BTCUSDT/2026-06.parquet     34000  2026-06

## Tardis 2024 raw days

In [ ]:
@section('tardis')
def _():
    rows = []
    for p in sorted(BASE.glob('TGNN_Tardis_Binance_Spot_*_v1')):
        rj = p / 'reports' / 'audit_report.json'; raw = list((p / 'raw').glob('*trades*.csv.gz')) if (p / 'raw').exists() else []
        rows.append({'folder': p.name.replace('TGNN_Tardis_Binance_Spot_', '').replace('_v1', ''), 'audit_report': rj.exists(), 'raw_trades_file': bool(raw), 'raw_MB': round(sum(q.stat().st_size for q in (p / 'raw').glob('*')) / 1e6) if (p / 'raw').exists() else 0})
    T = pd.DataFrame(rows); say('[Tardis 2024 raw days available for rebuilding Layer 1 bars]'); say((T).to_string()); RES['tardis_days'] = rows
    l1 = RES.get('layer1_end_day'); say(f'  Layer 1 ends {l1}; Tardis days after that date: ' + ', '.join(r['folder'] for r in rows if r['folder'].split('_')[-1] > (l1 or '9999') and r['raw_trades_file']))

[Tardis 2024 raw days available for rebuilding Layer 1 bars]
                folder  audit_report  raw_trades_file  raw_MB
0   BTCUSDT_2024-05-01          True             True     272
1   BTCUSDT_2024-06-01          True             True      70
2   BTCUSDT_2024-07-01          True             True     141
3   BTCUSDT_2024-08-01          True             True     309
4   BTCUSDT_2024-09-01          True             True     207
5   BTCUSDT_2024-10-01          True             True     282
6   ETHUSDT_2024-05-01          True             True     230
7   ETHUSDT_2024-06-01          True             True      82
8   ETHUSDT_2024-07-01          True             True     110
9   ETHUSDT_2024-08-01          True             True     288
10  ETHUSDT_2024-09-01          True             True     191
11  ETHUSDT_2024-10-01          True             True     266
  Layer 1 ends 2024-07-29; Tardis days after that date: BTCUSDT_2024-08-01, BTCUSDT_2024-09-01, BTCUSDT_2024-10-01, ETHUSDT_2024-08-0

## Saved report and share block

In [ ]:
rep = {'results': RES, 'failed_sections': FAILS, 'lines': LINES}; (OUT / 'audit_v1.json').write_text(json.dumps(rep, indent=1, default=str))
txt = '\n'.join(LINES); sb = ['==== AUDIT SHARE BLOCK (copy everything between the two ==== lines) ====', f'failed sections: {FAILS or "none"}', txt[:14000] + (' ...[cut]' if len(txt) > 14000 else ''), '==== END ====']
print('\n'.join(sb[:2]), '\n(the full text is in data_audit_v1/audit_share_block.txt and printed above)'); (OUT / 'audit_share_block.txt').write_text('\n'.join(sb))

==== AUDIT SHARE BLOCK (copy everything between the two ==== lines) ====
failed sections: none 
(the full text is in data_audit_v1/audit_share_block.txt and printed above)


13789

In [ ]:
print(open('/content/drive/MyDrive/research datasets/data_audit_v1/audit_share_block.txt').read())

==== AUDIT SHARE BLOCK (copy everything between the two ==== lines) ====
failed sections: none
[Layer 1] daily files: 180
  symbols ['BTCUSDT', 'ETHUSDT']; days per symbol {'BTCUSDT': 90, 'ETHUSDT': 90}; range 2024-05-01 -> 2024-07-29
  days with rows != 1440: 0 | minute gaps: 0 | duplicate timestamps: 0 | NaN cells: 0 | inf: 0 | OHLC violations: 0 | negative volume: 0 | zero-volume minutes: 0
  hash mismatches: 180 | validation_passed False: 0
  BTCUSDT: monthly summary
         minutes  vol_mean  ret_std  trades_mean  tfi_mean  large_mean
m                                                                     
2024-05    44640  21.17005  0.00064    906.53156  -0.03220    55.14014
2024-06    43200  16.13005  0.00051    818.09382  -0.02265    37.22347
2024-07    41760  20.64767  0.00072   1238.40802  -0.02999    46.98381
  ETHUSDT: monthly summary
         minutes   vol_mean  ret_std  trades_mean  tfi_mean  large_mean
m                                                                     